In [ ]:
import pandas as pd
from IPython.display import Audio, IFrame
from tqdm import tqdm

from utils.utils import get_df, get_lens, join_dfs
from utils.viz import visualize_annotations

pd.set_option("display.max_colwidth", None)

In [ ]:
TEXT = "../data/01_Text"
text_corpus = {
    "m": TEXT + "/First_group/1_M.csv",
    "c": TEXT + "/First_group/2_C.csv",
    "g": TEXT + "/First_group/3_G.csv",
    "a": TEXT + "/Second_group/1_A.csv",
    "r": TEXT + "/Second_group/2_R.csv",
    "n": TEXT + "/Second_group/3_N.csv",
}

# Text
dfs = {k: get_df(file) for k, file in text_corpus.items()}

# TEXTS agregats
joined = pd.read_csv(TEXT + "/text_with_mmlabel.csv")
joined["len"] = joined.text_clean.str.len()

# Filtre mateixa anotació text i multimodal (vídeo) + agreement anotadors
joined = joined[(joined.sexist_text == joined.sexist_video) & (joined.sexist_soft.isin((0, 1)))].sort_values(by="len")
IDS = joined.id.tolist()

In [ ]:
dfs_sexist = {}
dfs_no_sexist = {}
for persona, df in dfs.items():
    print(persona)
    dfs_sexist[persona] = get_lens(df, IDS, "SEXIST")
    dfs_no_sexist[persona] = get_lens(df, IDS, "NO-SEXIST")

In [ ]:
# row = dfs["c"].iloc[0]
# row.label_clean

In [ ]:
ID_COL = "video_id"
binary_cols = [
    "sexist",
    # "irony",
    # "humor",
    # "implicit sexism",
    # "stereotypes",
    # "inequality",
    # "discrimination",
    # "objectification",
    # "critique",
    # "reported_sexism",
    # "sexist_irony",
    # "sexist_humor",
    # "no_sexist_irony",
    # "no_sexist_humor",
]
num_cols = [
    "span_len",
    "span_num",
]
label_col = "label_clean"
cols = binary_cols + num_cols + [label_col]

cols_left = [ID_COL, "text_clean"] + cols
cols_right = [ID_COL] + cols


def master_join(dfs):
    df = join_dfs(dfs, cols_left, cols_right, ID_COL, binary_cols, num_cols, label_col)
    # df["text_clean"] = df.text.apply(clean_text).apply(clean_one_speaker_tag)
    df["len"] = df.text_clean.str.len()
    df = df.sort_values(by=["span_len", "len"])
    # df[["id", "text_clean"] + binary_cols + [c + "_soft" for c in binary_cols] + num_cols].to_csv(
    #     os.path.join(TEXT, "text.csv"), index=False
    # )
    return df


df_sexist = master_join(dfs_sexist)
df_no_sexist = master_join(dfs_no_sexist)

# Tria de textos

In [ ]:
# nombre de textos sexistes i no sexistes -> total és el doble
NUM = 20
NUM = 10

## Sexistes

Descartats:
- 95 meh
- 149 (per vocabulari estrany)
- 297 (molt llarg)
- 385 meh
- 20 outlier de llargada

Editar: 
- 151 última pregunta és una afirmació
- 201 posar cometes
- 44 afegir qui parla
- 62 afegir qui parla
- 76 afegir qui parla

In [ ]:
# Ens quedem amb els més curts o amb spans més curts
df = df_sexist
if NUM == 20:
    short_spans = set(df.iloc[:14].index)
    short_lens = set(df.sort_values(by=["len", "span_len"]).iloc[:15].index)
elif NUM == 10:
    short_spans = set(df.iloc[:7].index)
    short_lens = set(df.sort_values(by=["len", "span_len"]).iloc[:7].index)
descartats = {7, 24, 41, 82}
idxs = list(short_spans.union(short_lens) - descartats)
idxs_sexist = idxs
len(idxs)

## No-Sexistes

Descartats: 
- 13 meh
- 18 llarguet
- 216 amb 2 d'amor romántico fem
- 63 meh


Posar exemples abans d'experiment. Crítica no és sexista

In [ ]:
# Ens quedem amb els més curts + potser algun de llarg?
df = df_no_sexist
if NUM == 20:
    short_lens = set(df.sort_values(by=["len", "span_len"]).iloc[:24].index)
elif NUM == 10:
    short_lens = set(df.sort_values(by=["len", "span_len"]).iloc[:12].index)
descartats = {3, 21, 46, 58}
idxs = list(short_lens - descartats)
idxs_no_sexist = idxs
len(idxs)

## Selecció (pels dos casos)

In [ ]:
# Els iterem un per un, escoltem que tinguin sentit mentre observem les anotacions amb pdf
get_idx = iter(idxs)

In [ ]:
idx = next(get_idx)
video_id = int(df.loc[idx].id.split("_")[1])
print(video_id)

In [ ]:
row = df.loc[idx]
anotacions = row.label_clean
pdf_file = f"{video_id}.pdf"
visualize_annotations(row.text_clean, row.label_clean, file=pdf_file)
try:
    display(Audio(f"../data/mp3_files/{video_id}.mp3", autoplay=True))
except ValueError:
    display(Audio(f"../data/mp3_files/{video_id}_.mp3", autoplay=True))
IFrame(src="pdfs_anotacions/" + pdf_file, width=800, height=400)

# Unió

In [ ]:
df = pd.concat([df_sexist.loc[idxs_sexist], df_no_sexist.loc[idxs_no_sexist]])

In [ ]:
df[df.sexist == 1].len.hist()
df[df.sexist == 0].len.hist(alpha=0.5)

In [ ]:
df.groupby(by="sexist")[["len", "span_num", "span_len"]].agg(["mean", "std"])

In [ ]:
# aproximant mil caràcters per minut, es tardarà això a llegir aquestes frases
df.len.sum() / 1000

Si hi sumem estimulated recall potser queda massa llarg tot plegat

- 40 textos -> 17 min llegint + setup + recall
- 20 textos -> 7 min llegint + setup + recall (més raonable, crec)

In [ ]:
df[["id", "text_clean"]].to_csv("../data/chosen_data.csv", index=False)
df[["id", "sexist", "text_clean"]]

# Label clean

In [ ]:
row = df.iloc[0]
for ann in row["label_clean"]:
    print("---", ann["text"])
    break

In [ ]:
for i in tqdm(range(3)):
    row = df.iloc[i]
    anotacions = row.label_clean
    visualize_annotations(row.text_clean, row.label_clean, file=f"{i}.pdf")